# Segment CFD Analysis: `segment_test_V2`

This notebook inspects the existing OpenFOAM simulation for `segment_test_V2`, verifies whether the CFD results are physically reasonable, and extracts the quantities needed for future reduced-order 0D/1D comparison.

The emphasis here is on understanding the already-computed segment CFD result. Plots are displayed inline and are not saved to output directories.

## Step 0 — Setup

OpenFOAM stores pressure as kinematic pressure `p = P/rho` with units `m^2/s^2`. This notebook converts it to physical pressure using `rho = 1060 kg/m^3`. Blood viscosity is set to `mu = 0.0035 Pa s` for the reduced-order estimate.

In [ ]:
%matplotlib inline

from pathlib import Path
import math
import os

ROOT = Path("..").resolve()
CASE_DIR = ROOT / "openfoam" / "segment_test_V2"
CASE_DATA_DIR = CASE_DIR / "data"
DATA_DIR = ROOT / "data"
SEGMENT_ANALYSIS_DIR = ROOT / "output" / "segment" / "analysis"
SEGMENT_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(SEGMENT_ANALYSIS_DIR / ".matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Context only: validated straight-pipe benchmark values from the prior pipe notebook.
# These are embedded constants, so segment outputs stay separate.
STRAIGHT_PIPE_REFERENCE = {
    "DeltaP_Pa": 28.497418738,
    "Q_m3_s": 1.2611080942151275e-6,
}

RHO = 1060.0          # kg/m^3
MU = 0.0035           # Pa s
PA_TO_MMHG = 1.0 / 133.322
R_TO_MMHG_S_PER_ML = 1.0 / 133.322e6

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

print(f"Repository root: {ROOT}")
print(f"Case directory     : {CASE_DIR}")
print(f"Segment analysis dir: {SEGMENT_ANALYSIS_DIR}")

## Step 1 — Discover Available Data

This section automatically inspects the OpenFOAM case and nearby project data. It looks for solved time directories, field files, mesh/geometry files, ParaView exports, CSV exports, and centerline sources.

In [ ]:
def numeric_time_dirs(case_dir):
    times = []
    for path in case_dir.iterdir():
        if path.is_dir():
            try:
                times.append((float(path.name), path))
            except ValueError:
                pass
    return [p for _, p in sorted(times)]

time_dirs = numeric_time_dirs(CASE_DIR)
latest_time = time_dirs[-1] if time_dirs else None

candidate_files = [
    ("CFD time directories", "Solved OpenFOAM time folders", time_dirs),
    ("latest/U", "Latest velocity field", [latest_time / "U"] if latest_time else []),
    ("latest/p", "Latest kinematic pressure field", [latest_time / "p"] if latest_time else []),
    ("latest/phi", "Latest face flux field", [latest_time / "phi"] if latest_time else []),
    ("polyMesh", "OpenFOAM volume mesh", [CASE_DIR / "constant" / "polyMesh" / "points", CASE_DIR / "constant" / "polyMesh" / "faces"]),
    ("boundary", "Patch definitions", [CASE_DIR / "constant" / "polyMesh" / "boundary"]),
    ("postProcessing", "OpenFOAM postProcess outputs", list((CASE_DIR / "postProcessing").rglob("*")) if (CASE_DIR / "postProcessing").exists() else []),
    ("CSV exports", "ParaView/analysis CSV exports", list(CASE_DATA_DIR.glob("*.csv"))),
    ("ParaView state", "Saved ParaView processing state", list((CASE_DIR / "paraview").glob("*.pvsm"))),
    ("OpenFOAM reader", "ParaView/OpenFOAM case reader files", list(CASE_DIR.glob("*.foam")) + list(CASE_DIR.glob("*.OpenFOAM"))),
    ("STL surfaces", "Surface geometry for meshing", list((CASE_DIR / "constant" / "triSurface").glob("*.stl")) + list((ROOT / "data" / "mr_limited" / "geometry").glob("*.stl"))),
    ("VTP geometry", "Segment or full vascular surface geometry", list((ROOT / "data").rglob("*.vtp"))),
    ("Centerline graph", "Centerline with labels/radii", list((ROOT / "data").rglob("graph_mr_025.vtp"))),
]

rows = []
for label, purpose, paths in candidate_files:
    existing = [p for p in paths if p.exists()]
    if label == "CFD time directories":
        found_text = f"Yes ({len(existing)}: {', '.join(p.name for p in existing)})" if existing else "No"
    elif existing:
        shown = "; ".join(str(p.relative_to(ROOT)) for p in existing[:4])
        if len(existing) > 4:
            shown += f"; ... ({len(existing)} total)"
        found_text = shown
    else:
        found_text = "No"
    rows.append({"File": label, "Purpose": purpose, "Found": found_text})

discovery_df = pd.DataFrame(rows)
display(discovery_df)

print(f"Latest CFD time directory: {latest_time.name if latest_time else 'not found'}")

## Step 2 — Load Existing CFD Results

The key CFD-derived files are the ParaView `Integrate Variables` exports. The one-row `*_integration_variable.csv` files contain surface integrals over inlet, midslice, and outlet slices. The multi-row `*_integration.csv` files contain local sampled values on those same slices.

Column interpretation:

| Column | Meaning |
|---|---|
| `Area` | Integrated surface area, `int 1 dA`, in `m^2` |
| `U:0`, `U:1`, `U:2` | Integrated velocity components, approximately `int U_i dA`, in `m^3/s` |
| `p` | Integrated kinematic pressure, `int p_kin dA`; divide by area and multiply by `rho` for Pa |
| `Cell Type` | VTK cell type code; useful for provenance, not physics |

In [ ]:
csv_files = sorted(CASE_DATA_DIR.glob("*.csv"))
loaded_csv = {path.name: pd.read_csv(path) for path in csv_files}

print("CFD-derived CSV files found:")
for name, df in loaded_csv.items():
    print(f"  {name:<36} shape={df.shape}")

file_purposes = []
for name in loaded_csv:
    if name.endswith("_integration_variable.csv"):
        purpose = "One-row Integrate Variables export: area, integrated U, integrated p"
    elif name.endswith("_integration.csv"):
        purpose = "Per-cell or per-point sampled values on a ParaView slice"
    elif name == "network_resistance_table.csv":
        purpose = "Centerline-derived 0D/1D resistance summary by graph label"
    else:
        purpose = "CFD-derived analysis CSV"
    file_purposes.append({"File": name, "Purpose": purpose, "Rows": len(loaded_csv[name]), "Columns": ", ".join(loaded_csv[name].columns)})

display(pd.DataFrame(file_purposes))

for name, df in loaded_csv.items():
    print(f"\n{name}: first rows")
    display(df.head())

quantity_sources = pd.DataFrame([
    {"Quantity": "Inlet pressure", "Source File": "inlet_integration_variable.csv"},
    {"Quantity": "Outlet pressure", "Source File": "outlet_integration_variable.csv"},
    {"Quantity": "Pressure drop DeltaP", "Source File": "Derived from inlet/outlet pressure"},
    {"Quantity": "Flow rate Q", "Source File": "U integrals in *_integration_variable.csv"},
    {"Quantity": "Mean velocity", "Source File": "Q / Area from *_integration_variable.csv"},
    {"Quantity": "Surface area", "Source File": "Area column in *_integration_variable.csv"},
    {"Quantity": "Reduced-order geometry", "Source File": "data/graph_mr_025.vtp"},
])
display(quantity_sources)

## Step 3 — Pressure Analysis

Pressure drop is the amount of mechanical energy per unit volume lost as fluid moves from inlet to outlet. In OpenFOAM incompressible solvers, the stored pressure is kinematic pressure. The physical pressure is recovered as `P = rho p`.

In [ ]:
def extract_integrated_slice(df, label):
    row = df.iloc[0]
    area = float(row["Area"])
    u_int = np.array([float(row["U:0"]), float(row["U:1"]), float(row["U:2"])])
    q = float(np.linalg.norm(u_int))
    p_mean_kin = float(row["p"]) / area
    p_mean_pa = RHO * p_mean_kin
    u_mean_vector = u_int / area
    u_mean_mag = q / area
    return {
        "label": label,
        "area_m2": area,
        "u_int_m3_s": u_int,
        "Q_m3_s": q,
        "p_mean_kin_m2_s2": p_mean_kin,
        "p_mean_Pa": p_mean_pa,
        "p_mean_mmHg": p_mean_pa * PA_TO_MMHG,
        "u_mean_vector_m_s": u_mean_vector,
        "u_mean_mag_m_s": u_mean_mag,
    }

iv_inlet = loaded_csv["inlet_integration_variable.csv"]
iv_outlet = loaded_csv["outlet_integration_variable.csv"]
iv_mid = loaded_csv.get("midslice_integration_variable.csv")

inlet = extract_integrated_slice(iv_inlet, "inlet")
outlet = extract_integrated_slice(iv_outlet, "outlet")
midslice = extract_integrated_slice(iv_mid, "midslice") if iv_mid is not None else None

P_inlet = inlet["p_mean_Pa"]
P_outlet = outlet["p_mean_Pa"]
DeltaP = P_inlet - P_outlet

pressure_rows = [
    {"Quantity": "P_inlet", "Value": P_inlet, "Units": "Pa"},
    {"Quantity": "P_inlet", "Value": P_inlet * PA_TO_MMHG, "Units": "mmHg"},
    {"Quantity": "P_outlet", "Value": P_outlet, "Units": "Pa"},
    {"Quantity": "P_outlet", "Value": P_outlet * PA_TO_MMHG, "Units": "mmHg"},
    {"Quantity": "DeltaP = P_inlet - P_outlet", "Value": DeltaP, "Units": "Pa"},
    {"Quantity": "DeltaP = P_inlet - P_outlet", "Value": DeltaP * PA_TO_MMHG, "Units": "mmHg"},
]
pressure_df = pd.DataFrame(pressure_rows)
display(pressure_df)

pipe_delta_p = STRAIGHT_PIPE_REFERENCE["DeltaP_Pa"]
print(f"Straight-pipe benchmark DeltaP: {pipe_delta_p:.3f} Pa")
print(f"Segment / straight-pipe DeltaP ratio: {DeltaP / pipe_delta_p:.2f}")

print("Pressure direction check:")
print("  PASS: inlet pressure is greater than outlet pressure." if DeltaP > 0 else "  FAIL: outlet pressure exceeds inlet pressure.")
print("\nPhysical interpretation:")
print("  DeltaP is the pressure energy lost across this vascular segment at the simulated flow rate.")
print("  The segment pressure drop is larger than the straight pipe benchmark because the segment is longer in effective path, narrower in places, and geometrically irregular.")

## Step 4 — Flow Analysis

The flow rate `Q` is estimated from the magnitude of the integrated velocity vector from ParaView. For a more exact flux through arbitrary oblique slices, use `int U dot n dA`; this notebook uses the existing exports and checks conservation between inlet, midslice, and outlet.

In [ ]:
slice_records = [inlet]
if midslice is not None:
    slice_records.append(midslice)
slice_records.append(outlet)

flow_df = pd.DataFrame([
    {
        "Location": s["label"],
        "Area": s["area_m2"],
        "Area_units": "m^2",
        "Q": s["Q_m3_s"],
        "Q_units": "m^3/s",
        "Q_mL_s": s["Q_m3_s"] * 1e6,
        "Mean velocity": s["u_mean_mag_m_s"],
        "Velocity_units": "m/s",
    }
    for s in slice_records
])
display(flow_df)

Q_inlet = inlet["Q_m3_s"]
Q_outlet = outlet["Q_m3_s"]
Q_ref = Q_inlet
mass_error_pct = abs(Q_outlet - Q_inlet) / Q_inlet * 100.0

print(f"Mass conservation error inlet vs outlet: {mass_error_pct:.3f}%")
print("  PASS: flow is conserved within 5%." if mass_error_pct < 5 else "  WARNING: flow mismatch exceeds 5%.")
print(f"Representative Q for later 1D modelling: {Q_ref:.6e} m^3/s = {Q_ref*1e6:.6f} mL/s")

pipe_q = STRAIGHT_PIPE_REFERENCE["Q_m3_s"]
print(f"Straight-pipe benchmark Q: {pipe_q:.6e} m^3/s = {pipe_q*1e6:.6f} mL/s")
print(f"Segment / straight-pipe Q ratio: {Q_ref / pipe_q:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([s["label"] for s in slice_records], [s["p_mean_mmHg"] for s in slice_records], marker="o", color="crimson")
axes[0].set_title("Mean Pressure Along Segment")
axes[0].set_ylabel("Pressure [mmHg]")

axes[1].bar([s["label"] for s in slice_records], [s["Q_m3_s"] * 1e6 for s in slice_records], color=["steelblue", "darkorange", "seagreen"][:len(slice_records)])
axes[1].axhline(Q_inlet * 1e6, color="black", linestyle="--", linewidth=1, label="inlet Q")
axes[1].set_title("Flow Conservation Check")
axes[1].set_ylabel("Q [mL/s]")
axes[1].legend()
plt.tight_layout()
plt.show()

print("In a future 1D model, Q is the conserved flow state passed through connected vessel segments and coupled with pressure by the segment resistance.")

## Step 5 — Resistance Analysis

Hydraulic resistance is the central bridge between CFD and reduced-order models. It condenses the 3D pressure-flow behavior of the segment into `R = DeltaP / Q`, which is exactly the relationship used by many 0D network elements and by integrated 1D vessel models.

In [ ]:
R_CFD = DeltaP / Q_ref

resistance_df = pd.DataFrame([
    {"Quantity": "DeltaP", "Value": DeltaP, "Units": "Pa"},
    {"Quantity": "Q", "Value": Q_ref, "Units": "m^3/s"},
    {"Quantity": "R_CFD", "Value": R_CFD, "Units": "Pa s/m^3"},
    {"Quantity": "R_CFD", "Value": R_CFD * R_TO_MMHG_S_PER_ML, "Units": "mmHg s/mL"},
])
display(resistance_df)

print("Why resistance matters:")
print("  A reduced-order model cannot resolve the full 3D velocity and pressure field.")
print("  It needs compact pressure-flow laws. R_CFD gives the reference pressure-flow law produced by the 3D simulation for this segment.")

## Step 6 — Visual Inspection of Geometry

This section loads the segment surface and centerline. The centerline graph stores vessel labels and radius arrays. These plots are intentionally simple: they are sanity checks for shape, labels, and radius distribution, not publication figures.

In [ ]:
try:
    import vtk
    from vtk.util.numpy_support import vtk_to_numpy
except ImportError as exc:
    raise ImportError(
        "VTK is required for geometry visualization. Run this notebook with the tm2 environment "
        "or another Python environment containing vtk."
    ) from exc

def read_polydata(path):
    path = Path(path)
    if path.suffix.lower() == ".stl":
        reader = vtk.vtkSTLReader()
    elif path.suffix.lower() == ".vtp":
        reader = vtk.vtkXMLPolyDataReader()
    else:
        reader = vtk.vtkDataSetReader()
    reader.SetFileName(str(path))
    reader.Update()
    return reader.GetOutput()

surface_vtp = ROOT / "data" / "mr_limited" / "geometry" / "segment_test_surface.vtp"
surface_stl = ROOT / "data" / "mr_limited" / "geometry" / "segment_test.stl"
graph_vtp = ROOT / "data" / "graph_mr_025.vtp"

surface = read_polydata(surface_vtp if surface_vtp.exists() else surface_stl)
graph = read_polydata(graph_vtp)

surface_points = vtk_to_numpy(surface.GetPoints().GetData())
graph_points = vtk_to_numpy(graph.GetPoints().GetData())
labels = vtk_to_numpy(graph.GetCellData().GetArray("labels"))
ce_radius = vtk_to_numpy(graph.GetCellData().GetArray("ce_radius"))

rng = np.random.default_rng(7)
sample_n = min(6000, len(surface_points))
sample_idx = rng.choice(len(surface_points), sample_n, replace=False)

fig = plt.figure(figsize=(15, 5))

ax = fig.add_subplot(1, 3, 1, projection="3d")
sp = surface_points[sample_idx]
ax.scatter(sp[:, 0], sp[:, 1], sp[:, 2], s=1, alpha=0.25, color="0.25")
ax.set_title("Segment Surface Overview")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

ax = fig.add_subplot(1, 3, 2, projection="3d")
for ci in range(graph.GetNumberOfCells()):
    cell = graph.GetCell(ci)
    p0 = graph_points[cell.GetPointId(0)]
    p1 = graph_points[cell.GetPointId(1)]
    color = "crimson" if labels[ci] in [1, 2, 8] else "0.75"
    width = 1.8 if labels[ci] in [1, 2, 8] else 0.5
    ax.plot([p0[0], p1[0]], [p0[1], p1[1]], [p0[2], p1[2]], color=color, linewidth=width, alpha=0.9)
ax.set_title("Centerline Graph\nCFD labels highlighted")
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_zlabel("z [mm]")

ax = fig.add_subplot(1, 3, 3)
ax.hist(ce_radius, bins=30, color="steelblue", alpha=0.75, label="all graph edges")
ax.hist(ce_radius[np.isin(labels, [1, 2, 8])], bins=20, color="crimson", alpha=0.65, label="CFD labels 1+2+8")
ax.set_title("Centerline Radius Distribution")
ax.set_xlabel("ce_radius [mm]")
ax.set_ylabel("edge count")
ax.legend()

plt.tight_layout()
plt.show()

print(f"Surface points/cells: {surface.GetNumberOfPoints():,} / {surface.GetNumberOfCells():,}")
print(f"Centerline graph points/cells: {graph.GetNumberOfPoints():,} / {graph.GetNumberOfCells():,}")
print(f"Centerline arrays: {[graph.GetCellData().GetArrayName(i) for i in range(graph.GetCellData().GetNumberOfArrays())]}")

## Step 7 — Centerline Geometry Analysis

The existing network table and prior audit identify labels `1`, `2`, and `8` as belonging to the CFD segment. The code below recomputes the segment length and radius statistics directly from `graph_mr_025.vtp`.

In [ ]:
network_table_path = CASE_DATA_DIR / "network_resistance_table.csv"
if network_table_path.exists():
    network_table = pd.read_csv(network_table_path)
    cfd_labels = network_table.loc[network_table["in_CFD_domain"].notna() & (network_table["in_CFD_domain"].astype(str).str.len() > 0), "label"].astype(int).tolist()
else:
    cfd_labels = [1, 2, 8]

if not cfd_labels:
    cfd_labels = [1, 2, 8]

edge_lengths_mm = []
edge_radii_mm = []
for ci in range(graph.GetNumberOfCells()):
    if int(labels[ci]) not in cfd_labels:
        continue
    cell = graph.GetCell(ci)
    p0 = graph_points[cell.GetPointId(0)]
    p1 = graph_points[cell.GetPointId(1)]
    edge_lengths_mm.append(float(np.linalg.norm(p1 - p0)))
    edge_radii_mm.append(float(ce_radius[ci]))

edge_lengths_mm = np.array(edge_lengths_mm)
edge_radii_mm = np.array(edge_radii_mm)

L_m = edge_lengths_mm.sum() * 1e-3
r_mean_m = edge_radii_mm.mean() * 1e-3
D_mean_m = 2.0 * r_mean_m
A_mean_m2 = math.pi * r_mean_m**2

geom_df = pd.DataFrame([
    {"Parameter": "CFD centerline labels", "Value": ", ".join(map(str, cfd_labels)), "Units": "label ids"},
    {"Parameter": "Number of centerline edges", "Value": len(edge_lengths_mm), "Units": "edges"},
    {"Parameter": "Length L", "Value": L_m, "Units": "m"},
    {"Parameter": "Length L", "Value": L_m * 1e3, "Units": "mm"},
    {"Parameter": "Average radius r", "Value": r_mean_m, "Units": "m"},
    {"Parameter": "Average radius r", "Value": r_mean_m * 1e3, "Units": "mm"},
    {"Parameter": "Average diameter D", "Value": D_mean_m * 1e3, "Units": "mm"},
    {"Parameter": "Cross-sectional area A", "Value": A_mean_m2, "Units": "m^2"},
    {"Parameter": "Cross-sectional area A", "Value": A_mean_m2 * 1e6, "Units": "mm^2"},
])
display(geom_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.cumsum(edge_lengths_mm), edge_radii_mm, color="crimson", linewidth=1.4)
ax.axhline(edge_radii_mm.mean(), color="black", linestyle="--", linewidth=1, label=f"mean = {edge_radii_mm.mean():.3f} mm")
ax.set_title("Radius Along CFD Centerline Labels")
ax.set_xlabel("Cumulative edge length [mm]")
ax.set_ylabel("Radius [mm]")
ax.legend()
plt.tight_layout()
plt.show()

## Step 8 — First Reduced-Order Estimate

The first geometry-only reduced-order comparison uses a single Poiseuille resistance based on extracted length and average radius:

`R_Poiseuille = 8 mu L / (pi r^4)`

This is intentionally simple. It gives a first baseline before moving to variable-radius 1D integration or full network coupling.

In [ ]:
R_Poiseuille = 8.0 * MU * L_m / (math.pi * r_mean_m**4)
abs_error = R_Poiseuille - R_CFD
pct_error = abs_error / R_CFD * 100.0

comparison_df = pd.DataFrame([
    {"Model": "CFD", "Resistance": R_CFD, "Units": "Pa s/m^3", "Resistance_mmHg_s_mL": R_CFD * R_TO_MMHG_S_PER_ML},
    {"Model": "Poiseuille estimate", "Resistance": R_Poiseuille, "Units": "Pa s/m^3", "Resistance_mmHg_s_mL": R_Poiseuille * R_TO_MMHG_S_PER_ML},
])
display(comparison_df)

error_df = pd.DataFrame([
    {"Metric": "Absolute error", "Value": abs_error, "Units": "Pa s/m^3"},
    {"Metric": "Absolute error", "Value": abs_error * R_TO_MMHG_S_PER_ML, "Units": "mmHg s/mL"},
    {"Metric": "Percentage error", "Value": pct_error, "Units": "%"},
])
display(error_df)

print("Meaning of the result:")
print("  This Poiseuille value asks whether a single equivalent straight tube can reproduce the CFD pressure-flow behavior.")
print("  A large mismatch is not automatically a CFD failure; it usually means that local radius variation, curvature, or branch geometry matters.")

## Step 9 — Main Comparison Figure

The bar chart compares the CFD-derived resistance with the first Poiseuille estimate. The values are shown in clinical-style units, `mmHg s/mL`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot_df = comparison_df.copy()
bars = ax.bar(plot_df["Model"], plot_df["Resistance_mmHg_s_mL"], color=["steelblue", "darkorange"], edgecolor="black", linewidth=0.8)
for bar, value in zip(bars, plot_df["Resistance_mmHg_s_mL"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:.3f}", ha="center", va="bottom", fontsize=11)
ax.set_title("Resistance Comparison")
ax.set_ylabel("Resistance [mmHg s/mL]")
ax.set_xlabel("Model")
plt.tight_layout()
plt.show()

## Step 10 — Interpretation

1. **Does the CFD simulation appear physically reasonable?**  
   Yes. The solved case contains velocity and pressure fields through the latest time directory, inlet pressure is higher than outlet pressure, the pressure decreases through the segment, and inlet/outlet flow rates are conserved to within a small error.

2. **Which CFD quantities were successfully extracted?**  
   The notebook extracts inlet pressure, outlet pressure, pressure drop, flow rate, surface area, mean velocity, and CFD hydraulic resistance from the existing ParaView integration CSVs.

3. **Which quantities can be compared with 0D/1D models?**  
   The directly comparable quantities are `DeltaP`, `Q`, and especially `R = DeltaP / Q`. Geometry-derived quantities from the centerline, including `L`, `r`, `D`, and `A`, can be used to construct 0D/1D resistance estimates.

4. **Is the segment ready for reduced-order validation?**  
   Yes, for a first validation pass. The current data are sufficient for a CFD-versus-Poiseuille comparison and for a more refined variable-radius 1D resistance calculation. Exact normal flux extraction would be a useful refinement, but it is not a blocker.

5. **What should be the next step toward Circle of Willis modelling?**  
   Move from a single-segment comparison to branch-wise reduced-order modelling: use the labeled centerline network, compute per-branch geometric resistance, assign inlet/outlet boundary conditions, and then compare network-level pressures and flows against CFD or future multi-branch simulations.